In [1]:
import pandas as pd
import numpy as np
import torch as th
import os

from frag_gnn.runner import load_config, init_dataset

In [2]:
os.chdir("/h/adamo/frag-gnn/")

In [3]:
proc_dp = "data/proc/nist20mona23v3"
spec_df = pd.read_pickle(f"{proc_dp}/spec_df.pkl")
mol_df = pd.read_pickle(f"{proc_dp}/mol_df.pkl")
ann_df = pd.read_pickle(f"{proc_dp}/ann_df.pkl")

In [44]:
def ann_mz_rel_diffs_fn(row):
    exact_mzs = np.array(row["ann_exact_mzs"], dtype=np.float32)
    peak_mzs = np.array(row["ann_peak_mzs"], dtype=np.float32)
    return np.abs(exact_mzs - peak_mzs) / np.maximum(exact_mzs, 200.)

mz_rel_diffs = ann_df.apply(ann_mz_rel_diffs_fn, axis=1)
max_mz_rel_diffs = mz_rel_diffs.apply(np.max)
ann_df.loc[:,"ann_max_mz_rel_diff"] = max_mz_rel_diffs

In [4]:
split_dp = "data/split/nist20mona23v2_d4_wr00_inchikey"
train_ids_df_1 = pd.read_csv(f"{split_dp}/train_ids.csv")
val_ids_df_1 = pd.read_csv(f"{split_dp}/val_ids.csv")
test_ids_df_1 = pd.read_csv(f"{split_dp}/test_ids.csv")
# all_ids = np.concatenate([train_ids,val_ids,test_ids],axis=0)

In [5]:
template_fp = "config/template.yml"
custom_fp = "config/exp_graff_2/max_iso_2.yml"
config_d = load_config(template_fp, custom_fp)

In [6]:
# train_ds, val_ds, test_ds = init_dataset(config_d, splits=("train","val","test"))

In [7]:
# Check for both unmerged and merged:
# 1. (DONE) fraction of spectra with any annotations
# 2. fraction with mismatch annotations
# 3. (DONE) fraction with annotations and molecular weight < 1000 Da
# 4. (DONE) fraction or annotated spectra with normalized collision energy in {20, 35, 50}

In [50]:
def calculate_stats(split_ids_df, spec_df, mol_df, ann_df):

    split_spec_df = spec_df[spec_df["group_id"].isin(split_ids_df["group_id"])]
    split_mol_df = mol_df[mol_df["mol_id"].isin(split_ids_df["mol_id"])]
    split_ann_df = ann_df[ann_df["spec_id"].isin(split_spec_df["spec_id"])]
    split_spec_mol_df = split_spec_df.merge(split_mol_df, on="mol_id", how="inner")
    split_ann_mask = (split_spec_df["spec_id"].isin(ann_df["spec_id"])).to_numpy()
    split_ann_frac = split_ann_mask.mean()
    print(f"> Fraction of spectra with any annotations = {split_ann_frac}")
    split_light_mask = (split_spec_mol_df["exact_mw"] <= 1000.).to_numpy()
    split_ann_light_mask = split_ann_mask & split_light_mask
    split_light_frac = split_light_mask.mean()
    split_ann_light_frac = split_ann_light_mask.mean()
    print(f"> Fraction of spectra with molecular weight <= 1000 Da = {split_light_frac}")
    print(f"> Fraction of annotated spectra with molecular weight <= 1000 Da = {split_ann_light_frac}")
    split_prec_light_mask = (split_spec_mol_df["prec_mz"] <= 1000.).to_numpy()
    split_ann_prec_light_mask = split_ann_mask & split_prec_light_mask
    split_prec_light_frac = split_prec_light_mask.mean()
    split_ann_light_frac = split_ann_prec_light_mask.mean()
    print(f"> Fraction of spectra with precursor m/z <= 1000 Da = {split_prec_light_frac}")
    print(f"> Fraction of annotated spectra with precursor m/z <= 1000 Da = {split_ann_light_frac}")
    split_ce_mask = ((split_spec_df["nce"] == 20) | (split_spec_df["nce"] == 35) | (split_spec_df["nce"] == 50)).to_numpy()
    split_ann_ce_mask = split_ann_mask & split_ce_mask
    split_ce_frac = split_ce_mask.mean()
    split_ann_ce_frac = split_ann_ce_mask.mean()
    print(f"> Fraction of spectra with NCE in {{20, 35, 50}} = {split_ce_frac}")
    print(f"> Fraction of annotated spectra with NCE in {{20, 35, 50}} = {split_ann_ce_frac}")
    for t in [1e-5, 2e-5, 3e-5, 5e-5, 1e-4]:
        split_tol_mask = split_ann_df["ann_max_mz_rel_diff"] <= t
        split_tol_frac = split_tol_mask.mean()
        print(f"> Fraction of annotated spectra with relative peak mz differences <= {t} = {split_tol_frac}")    


In [51]:
split_to_split_ids_df = {
    "train": train_ids_df_1,
    "val": val_ids_df_1,
    "test": test_ids_df_1
}

for split, split_ids_df in split_to_split_ids_df.items():
    print(f">> Split = {split}")
    calculate_stats(split_ids_df, spec_df, mol_df, ann_df)

>> Split = train
> Fraction of spectra with any annotations = 0.9781136239678332
> Fraction of spectra with molecular weight <= 1000 Da = 1.0
> Fraction of annotated spectra with molecular weight <= 1000 Da = 0.9781136239678332
> Fraction of spectra with precursor m/z <= 1000 Da = 1.0
> Fraction of annotated spectra with precursor m/z <= 1000 Da = 0.9781136239678332
> Fraction of spectra with NCE in {20, 35, 50} = 0.17544616243229874
> Fraction of annotated spectra with NCE in {20, 35, 50} = 0.17096868301221477
> Fraction of annotated spectra with relative peak mz differences <= 1e-05 = 0.6438755859868894
> Fraction of annotated spectra with relative peak mz differences <= 2e-05 = 0.7164374590700721
> Fraction of annotated spectra with relative peak mz differences <= 3e-05 = 0.8562378831203357
> Fraction of annotated spectra with relative peak mz differences <= 5e-05 = 0.9874146550214943
> Fraction of annotated spectra with relative peak mz differences <= 0.0001 = 0.9893079681249068
>>

In [10]:
# measure performance on spectra without annotations
# measure performance on spectra with only (20, 35, 50) collision energies

In [12]:
graff_output_fp = "inference_outputs/v3_feb5/inchikey/graff/s0/test.pkl"
graff_output_d = th.load(graff_output_fp)

In [13]:
graff_output_d.keys()

dict_keys(['unique_id', 'smiles', 'mean_loss', 'total_loss', 'total_weight', 'spec_per_group', 'spec_per_mol', 'group_per_mol', 'true_mzs', 'true_logprobs', 'true_batch_idxs', 'true_prec_mzs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'loss', 'primary_loss', 'true_oos_e', 'cos_hun', 'cos_sim_0.01', 'true_oos_prob'])

In [14]:
graff_output_d["cos_sim_0.01"].mean()

tensor(0.6016)

In [18]:
graff_ids = graff_output_d["unique_id"].numpy()
graff_cos_sim = graff_output_d["cos_sim_0.01"].numpy()

In [19]:
test_ann_ids = test_ids_df_1[test_ids_df_1["spec_id"].isin(ann_df["spec_id"]).to_numpy()]["group_id"].to_numpy()
print(len(test_ann_ids), len(test_ids_df_1))

51515 52621


In [21]:
print(graff_cos_sim.mean())
print(graff_cos_sim[np.isin(graff_ids,test_ann_ids)].mean())
print(graff_cos_sim[~np.isin(graff_ids,test_ann_ids)].mean())

0.6015617
0.59809494
0.78109646
